Ejercicio 1: Análisis de Asociación

In [1]:
import pandas as pd

In [7]:
import numpy as np
from itertools import combinations

print("Librerías cargadas exitosamente.")

Librerías cargadas exitosamente.


Carga del Archivo de Datos Y Mostrar las primeras 5 lineas

In [9]:
df = pd.read_csv("https://raw.githubusercontent.com/karlaazuniga/2516662022_Zuniga_Karla/refs/heads/main/dataset/clave_H_asociacion.csv", sep=';')

In [10]:
print("--- PRIMERAS 5 FILAS DEL DATASET ---")
print(df.head())

print("\n--- DIMENSIONES DEL DATASET ---")
print(f"Total de registros (filas): {df.shape[0]}")
print(f"Total de columnas: {df.shape[1]}")

--- PRIMERAS 5 FILAS DEL DATASET ---
  transaccion_id,cliente_id,fecha,categoria,item,cantidad,canal
0  H-T0001,H-C0026,2026-04-10,Cloud,Backup_cloud,...           
1  H-T0001,H-C0026,2026-04-10,Productividad,CRM,2...           
2  H-T0001,H-C0026,2026-04-10,Educacion,Certifica...           
3  H-T0001,H-C0026,2026-04-10,Productividad,Notas...           
4  H-T0002,H-C0038,2026-03-16,Productividad,CRM,1...           

--- DIMENSIONES DEL DATASET ---
Total de registros (filas): 624
Total de columnas: 1


In [11]:
print("--- INFORMACIÓN GENERAL Y TIPOS DE DATOS ---")
df.info()

print("\n--- CONTEO DE VALORES NULOS POR COLUMNA ---")
print(df.isnull().sum())

print("\n--- FILAS DUPLICADAS IDENTIFICADAS ---")
duplicated_rows = df[df.duplicated(keep=False)]
print(f"Cantidad de filas duplicadas: {df.duplicated().sum()}")
if df.duplicated().sum() > 0:
    print(duplicated_rows)

--- INFORMACIÓN GENERAL Y TIPOS DE DATOS ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 624 entries, 0 to 623
Data columns (total 1 columns):
 #   Column                                                         Non-Null Count  Dtype 
---  ------                                                         --------------  ----- 
 0   transaccion_id,cliente_id,fecha,categoria,item,cantidad,canal  624 non-null    object
dtypes: object(1)
memory usage: 5.0+ KB

--- CONTEO DE VALORES NULOS POR COLUMNA ---
transaccion_id,cliente_id,fecha,categoria,item,cantidad,canal    0
dtype: int64

--- FILAS DUPLICADAS IDENTIFICADAS ---
Cantidad de filas duplicadas: 1
    transaccion_id,cliente_id,fecha,categoria,item,cantidad,canal
15   H-T0005,H-C0054,2026-04-29,Cloud,Almacenamient...           
623  H-T0005,H-C0054,2026-04-29,Cloud,Almacenamient...           


In [13]:
df_clean = df.drop_duplicates()

In [15]:
# Assuming the single column's name is the full header string from the initial load
column_name_full = df_clean.columns[0]
df_parsed_clean = df_clean[column_name_full].str.split(',', expand=True)

# Assign correct column names
df_parsed_clean.columns = ['transaccion_id', 'cliente_id', 'fecha', 'categoria', 'item', 'cantidad', 'canal']

print("DataFrame parsed and cleaned (head):")
print(df_parsed_clean.head())

basket = (df_parsed_clean.groupby(['transaccion_id', 'item']).size() > 0).astype(int).unstack(fill_value=0)

print("\nBasket DataFrame (head):")
print(basket.head())

DataFrame parsed and cleaned (head):
  transaccion_id cliente_id       fecha      categoria           item  \
0        H-T0001    H-C0026  2026-04-10          Cloud   Backup_cloud   
1        H-T0001    H-C0026  2026-04-10  Productividad            CRM   
2        H-T0001    H-C0026  2026-04-10      Educacion  Certificacion   
3        H-T0001    H-C0026  2026-04-10  Productividad          Notas   
4        H-T0002    H-C0038  2026-03-16  Productividad            CRM   

  cantidad   canal  
0        2     Web  
1        2     Web  
2        1     Web  
3        1     Web  
4        1  Tienda  


In [16]:
print("--- MATRIZ TRANSACCIONAL PROCESADA (PRIMERAS FILAS Y COLUMNAS) ---")
print(basket.iloc[:5, :5])

--- MATRIZ TRANSACCIONAL PROCESADA (PRIMERAS FILAS Y COLUMNAS) ---
item            Agenda  Almacenamiento  Antivirus  Backup_cloud  CRM
transaccion_id                                                      
H-T0001              0               0          0             1    1
H-T0002              0               0          0             0    1
H-T0003              0               1          0             1    0
H-T0004              0               1          0             0    0
H-T0005              0               1          0             0    0


In [17]:
print("\n--- RESUMEN DE COMPORTAMIENTO ---")
print(f"Número total de transacciones únicas analizadas: {basket.shape[0]}")
print(f"Número total de productos únicos evaluados: {basket.shape[1]}")


--- RESUMEN DE COMPORTAMIENTO ---
Número total de transacciones únicas analizadas: 200
Número total de productos únicos evaluados: 20


In [18]:
total_transacciones = len(basket)
item_supports = basket.mean()

In [20]:
records_reglas = []
items_unicos = basket.columns

In [21]:
for item_A, item_B in combinations(items_unicos, 2):
    # Calcular soporte conjunto (A y B)
    soporte_conjunto = ((basket[item_A] == 1) & (basket[item_B] == 1)).mean()

    # Filtrar por soporte mínimo de 0.05 (5%)
    if soporte_conjunto >= 0.05:
        # --- Regla: Si compra A -> Entonces compra B ---
        confianza_A_B = soporte_conjunto / item_supports[item_A]
        lift_A_B = confianza_A_B / item_supports[item_B]
        records_reglas.append({
            'Antecedente': item_A,
            'Consecuente': item_B,
            'Soporte': soporte_conjunto,
            'Confianza': confianza_A_B,
            'Lift': lift_A_B
        })

        # --- Regla: Si compra B -> Entonces compra A ---
        confianza_B_A = soporte_conjunto / item_supports[item_B]
        lift_B_A = confianza_B_A / item_supports[item_A]
        records_reglas.append({
            'Antecedente': item_B,
            'Consecuente': item_A,
            'Soporte': soporte_conjunto,
            'Confianza': confianza_B_A,
            'Lift': lift_B_A
        })

In [22]:
reglas_df = pd.DataFrame(records_reglas)
top_10_reglas = reglas_df.sort_values(by='Lift', ascending=False).head(10).reset_index(drop=True)

In [23]:
print("--- LAS 10 REGLAS DE ASOCIACIÓN MÁS RELEVANTES (ORDENADAS POR LIFT) ---")
print(top_10_reglas.to_string())

--- LAS 10 REGLAS DE ASOCIACIÓN MÁS RELEVANTES (ORDENADAS POR LIFT) ---
       Antecedente      Consecuente  Soporte  Confianza      Lift
0              VPN  Gestor_password    0.125   0.510204  2.616431
1  Gestor_password              VPN    0.125   0.641026  2.616431
2          Hosting          Dominio    0.105   0.466667  2.522523
3          Dominio          Hosting    0.105   0.567568  2.522523
4    Plan_familiar       Plan_video    0.140   0.651163  2.457218
5       Plan_video    Plan_familiar    0.140   0.528302  2.457218
6    Certificacion     Curso_python    0.140   0.549020  2.287582
7     Curso_python    Certificacion    0.140   0.583333  2.287582
8    Certificacion              CRM    0.050   0.196078  1.705030
9              CRM    Certificacion    0.050   0.434783  1.705030
